## 2D simulation of the optical lattice using the semi global scheme

In [1]:
#Import libraries
from blochstate1d_NEW import GroundBlochState, OLConstants # dimensions for 2d is specificed in blochstate1d_NEW.py. Check the file to make sure the dimensions are correct for the use case.
import numpy as np 
import time
from typing import cast
from quevolutio.core.domain import QuantumHilbertSpace, TimeGrid
from quevolutio.core.aliases import (  # isort: skip
    RVector,
    CTensors,
    CVectors
)
from crab_propagation_tools_semiglobal import OpticalLatticeHamiltonian
from quevolutio.propagators.semi_global import ApproximationBasis, SemiGlobal
from quevolutio.core.tdse import TDSE 
import utils.numerical as numerical
import OL_visualisation as vis
from pathlib import Path

In [2]:
#Set up the initial state
constants: OLConstants = OLConstants()
domain: QuantumHilbertSpace = QuantumHilbertSpace(
        num_dimensions=2,
        num_points=np.array([constants.num_pts, constants.num_pts]),
        position_bounds=np.array([[constants.lower_x_bound, constants.upper_x_bound], [constants.lower_x_bound, constants.upper_x_bound]]),
        constants=constants,
    )
state_idx: int = 0
initial_state = GroundBlochState().generate_bloch_state()
state_initial: RVector = cast(
        RVector, domain.normalise_state(initial_state)
    )

In [3]:
#Set up Hamiltonian, TDSE and time domain
hamiltonian = OpticalLatticeHamiltonian(domain) 
    
eigvals, eigvecs = np.linalg.eigh(GroundBlochState().H)
eigenvalue_min = np.min(eigvals)
eigenvalue_max = np.max(eigvals)

#Set up the TDSE
tdse: TDSE = TDSE(domain, hamiltonian)

# Set up the time domain.
time_domain: TimeGrid = TimeGrid(time_min=0.0, time_max=1.0, num_points=10001)
propagator = SemiGlobal(tdse, time_domain, order_m=10, order_k=10, tolerance=1e5, approximation=ApproximationBasis.NEWTONIAN)

print("Propagation Start")
start_time: float = time.time()
states: CTensors = propagator.propagate(
state_initial, diagnostics=True
    )
final_time: float = time.time()
print("Propagation Done")
runtime = final_time - start_time
print(f"Runtime: {runtime:.2f} seconds")
final_state = states[-1]

Propagation Start
Time Index: 0 	 Iteration: 1
Convergence: 1.11436e-04
Time Index: 1 	 Iteration: 1
Convergence: 1.94087e-16
Time Index: 2 	 Iteration: 1
Convergence: 2.41579e-16
Time Index: 3 	 Iteration: 1
Convergence: 2.38584e-16
Time Index: 4 	 Iteration: 1
Convergence: 2.38089e-16
Time Index: 5 	 Iteration: 1
Convergence: 2.54951e-16
Time Index: 6 	 Iteration: 1
Convergence: 2.52320e-16
Time Index: 7 	 Iteration: 1
Convergence: 2.50065e-16
Time Index: 8 	 Iteration: 1
Convergence: 2.54899e-16
Time Index: 9 	 Iteration: 1
Convergence: 2.50505e-16
Time Index: 10 	 Iteration: 1
Convergence: 2.42316e-16
Time Index: 11 	 Iteration: 1
Convergence: 2.46834e-16
Time Index: 12 	 Iteration: 1
Convergence: 2.48179e-16
Time Index: 13 	 Iteration: 1
Convergence: 2.33427e-16
Time Index: 14 	 Iteration: 1
Convergence: 2.36775e-16
Time Index: 15 	 Iteration: 1
Convergence: 2.45934e-16
Time Index: 16 	 Iteration: 1
Convergence: 2.51552e-16
Time Index: 17 	 Iteration: 1
Convergence: 2.41744e-16
Ti

In [4]:
#Plot the new states
# Set a common filename.
filename: str = "optical_lattice_2d"

    # Create directories for saving data if they do not exist.
for folder in (Path("data"), Path("figures"), Path("anims")):
    folder.mkdir(parents=True, exist_ok=True)

    # Save the propagated states.
np.save(f"data/{filename}.npy", states)

    # Plot the propagated states.
vis.plot_state_2D(
    states[0],
    domain,
    r"$\text{State} \; (T = 0.00)$",
    f"figures/{filename}_start.png",
    )
vis.plot_state_2D(
    states[-1],
    domain,
    rf"$\text{{State}} \; (T = {time_domain.time_axis[-1]:.2f})$",
    f"figures/{filename}_final.png",
    )


findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family ['Lato'] not found. Falling back to DejaVu Sans.
findfont: Font family 'Lato' not found.
findfont: Font family ['Lato'] not found. Falling back to DejaVu Sans.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family 'Lato' not found.
findfont: Font family ['Lato'] not found. Falling back to DejaVu Sans.
findfont: Font family 'Lato

In [5]:
norms: RVector = np.empty(time_domain.num_points, dtype=np.float64)

    # Calculate the position expectation values in of the states.
states_expectation_x: RVector = np.empty(time_domain.num_points, dtype=np.float64)
states_expectation_y: RVector = np.empty(time_domain.num_points, dtype=np.float64)

    # This is done in batches due to memory constraints.
batch: int = 1000
 # This is done in batches due to memory constraints.
batch: int = 1000

for i in range(0, time_domain.num_points, batch):
        # Calculate the current batch index.
        idx: int = min((i + batch), time_domain.num_points)

        # Calculate the norms of the states.
        norms[i:idx] = numerical.states_norms(states[i:idx], domain)

        # Calculate the position expectation values in x of the states.
        states_expectation_x[i:idx] = np.trapezoid(
            np.trapezoid(
                ((np.abs(states[i:idx]) ** 2) * domain.position_meshes[0]),
                dx=domain.position_deltas[0],
                axis=1,
            ),
            axis=1,
            dx=domain.position_deltas[1],
        )

        # Calculate the position expectation values in y of the states.
        states_expectation_y[i:idx] = np.trapezoid(
            np.trapezoid(
                ((np.abs(states[i:idx]) ** 2) * domain.position_meshes[1]),
                dx=domain.position_deltas[0],
                axis=1,
            ),
            axis=1,
            dx=domain.position_deltas[1],
        )

In [6]:
# Calculate the error from the exact position expectation values in x.
exact_expectation_x: RVector = 0.5 * (
        (time_domain.time_axis * np.cos(time_domain.time_axis))
        - np.sin(time_domain.time_axis)
    )
errors_x: RVector = np.abs(exact_expectation_x - states_expectation_x)

    # Calculate the error from the exact position expectation values in y.
exact_expectation_y: RVector = 0.5 * (
        (time_domain.time_axis * np.cos(time_domain.time_axis))
        - np.sin(time_domain.time_axis)
    )
errors_y: RVector = np.abs(exact_expectation_y - states_expectation_y)

    # Calculate the total magnitude error.
errors: RVector = np.sqrt((errors_x**2) + (errors_y**2))

    # Print simulation information.
print(f"Runtime \t\t: {(final_time - start_time):.5f} seconds")
print(f"Max Error \t\t: {np.max(errors):.5e}")
print(f"Max Norm Deviation \t: {np.max(np.abs(norms - norms[0])):.5e}")

Runtime 		: 574.62219 seconds
Max Error 		: 2.12958e-01
Max Norm Deviation 	: 3.56948e-04
